## 07-statement-svm-texts


In [1]:
import numpy as np
from sklearn import datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold

# 1. Загрузка данных
newsgroups = datasets.fetch_20newsgroups(
    subset='all',
    categories=['alt.atheism', 'sci.space']
)

X_texts = newsgroups.data   # список текстов
y = newsgroups.target       # метки классов (0 или 1)


# 2. Вычисление TF-IDF по ВСЕМ данным (как требуется в задании)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X_texts)  # разреженная матрица

print(f"Размерность признаков: {X.shape}")

# 3. Подбор параметра C с помощью GridSearchCV
# Сетка: 10^-5 ... 10^5
C_values = np.power(10.0, np.arange(-5, 6))
param_grid = {'C': C_values}

# Кросс-валидация с перемешиванием и random_state=241
cv = KFold(n_splits=5, shuffle=True, random_state=241)

# SVM с линейным ядром
svm = SVC(kernel='linear', random_state=241)

# Поиск по сетке (важно: scoring='accuracy')
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1
)
grid_search.fit(X, y)

# Анализ результатов: ищем минимальный C с максимальной точностью
results = grid_search.cv_results_
best_score = grid_search.best_score_
print(f"\nЛучшая точность на кросс-валидации: {best_score:.4f}")

# Вывод всех результатов для наглядности
print("\nРезультаты по всем C:")
for mean_score, params in zip(results['mean_test_score'], results['params']):
    print(f"C={params['C']:.5f} -> accuracy={mean_score:.4f}")

# Находим минимальный C среди тех, у которых точность равна максимальной
max_acc = max(results['mean_test_score'])
best_indices = np.where(results['mean_test_score'] == max_acc)[0]
best_C_values = [results['params'][i]['C'] for i in best_indices]
optimal_C = min(best_C_values)   # минимальный C с лучшим качеством
print(f"\nОптимальный C (минимальный с лучшей точностью): {optimal_C}")


Размерность признаков: (1786, 28382)

Лучшая точность на кросс-валидации: 0.9933

Результаты по всем C:
C=0.00001 -> accuracy=0.5526
C=0.00010 -> accuracy=0.5526
C=0.00100 -> accuracy=0.5526
C=0.01000 -> accuracy=0.5526
C=0.10000 -> accuracy=0.9502
C=1.00000 -> accuracy=0.9933
C=10.00000 -> accuracy=0.9933
C=100.00000 -> accuracy=0.9933
C=1000.00000 -> accuracy=0.9933
C=10000.00000 -> accuracy=0.9933
C=100000.00000 -> accuracy=0.9933

Оптимальный C (минимальный с лучшей точностью): 1.0


In [2]:
# 4. Обучаем SVM на всей выборке с найденным C
final_svm = SVC(kernel='linear', C=optimal_C, random_state=241)
final_svm.fit(X, y)

# 5. Находим 10 слов с наибольшим абсолютным весом
# coef_ имеет форму (1, n_features) для бинарной классификации
coefficients = final_svm.coef_.toarray().flatten()
abs_coef = np.abs(coefficients)
top_indices = np.argsort(abs_coef)[-10:]   # индексы 10 наибольших по модулю

# Получаем слова по индексам
feature_names = vectorizer.get_feature_names_out()
top_words = [feature_names[i] for i in top_indices]

# Сортируем лексикографически (в нижнем регистре они уже, но явно приводим)
top_words_sorted = sorted(top_words, key=str.lower)

print("\nТоп-10 слов по абсолютному весу (в лексикографическом порядке):")
print(top_words_sorted)

# Формируем ответ: слова через запятую, без пробелов
answer = ",".join(top_words_sorted)
print(f"\nОтвет для файла:\n{answer}")


Топ-10 слов по абсолютному весу (в лексикографическом порядке):
['atheism', 'atheists', 'bible', 'god', 'keith', 'moon', 'religion', 'sci', 'sky', 'space']

Ответ для файла:
atheism,atheists,bible,god,keith,moon,religion,sci,sky,space
